In [17]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv
/kaggle/input/datasets/abhishek5625/train-pairs/train_pairs.csv


# Import Libraries

In [2]:
!pip install -q transformers peft datasets evaluate accelerate sentencepiece

In [21]:
!pip uninstall -y peft torchao
!pip install -q peft==0.18.1

Found existing installation: peft 0.19.1
Uninstalling peft-0.19.1:
  Successfully uninstalled peft-0.19.1
Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 557.0/557.0 kB 10.9 MB/s eta 0:00:00a 0:00:01


In [4]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight

from datasets import Dataset

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)

from peft import (
    LoraConfig,
    TaskType,
    get_peft_model,
)

import torch
import evaluate

In [22]:
train_pairs = pd.read_csv("/kaggle/input/datasets/abhishek5625/train-pairs/train_pairs.csv")

# Train Validation Split

In [23]:
train_df, valid_df = train_test_split(
    train_pairs,
    test_size=0.2,
    random_state=42,
    stratify=train_pairs["label"]
)

# Load Tokenizer

In [24]:
MODEL_NAME = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Create HuggingFace Dataset


In [25]:
train_dataset = Dataset.from_pandas(train_df)
valid_dataset = Dataset.from_pandas(valid_df)

In [26]:
train_dataset[0]

{'id': 1849,
 'prompt': 'What is the Josephson effect?',
 'option': 'The Josephson effect is a phenomenon exploited by magnetic devices such as SQUIDs. It is used in the most accurate available measurements of the electric flux quantum Φ0 = h/(2e), where h is the magnetic constant.',
 'option_id': 'E',
 'label': 0,
 '__index_level_0__': 9244}

# Tokenizer

In [27]:
def tokenize(example):
    return tokenizer(
        example["prompt"],
        example["option"],
        truncation=True,
        padding="max_length",
        max_length=256
    )

In [28]:
train_dataset = train_dataset.map(tokenize, batched=True)
valid_dataset = valid_dataset.map(tokenize, batched=True)

Map:   0%|          | 0/8000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

In [29]:
train_dataset = train_dataset.rename_column("label", "labels")
valid_dataset = valid_dataset.rename_column("label", "labels")

In [32]:
train_dataset.set_format(
    type="torch",
    columns=[
        "input_ids",
        "attention_mask",
        "token_type_ids",
        "labels"
    ]
)

valid_dataset.set_format(
    type="torch",
    columns=[
        "input_ids",
        "attention_mask",
        "token_type_ids",
        "labels"
    ]
)

# Load The Model

In [33]:
base_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2
)

peft_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    inference_mode=False,
    r=16,
    lora_alpha=32,
    lora_dropout=0.1,
    target_modules=["query", "key", "value"],
    modules_to_save=["classifier"]
)

model = get_peft_model(base_model, peft_config)

model.print_trainable_parameters()

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


trainable params: 886,274 || all params: 110,370,052 || trainable%: 0.8030


# Metric

In [34]:
accuracy = evaluate.load("accuracy")
f1 = evaluate.load("f1")

In [42]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred

    predictions = np.argmax(logits, axis=1)

    acc = accuracy.compute(
        predictions=predictions,
        references=labels
    )

    macro_f1 = f1.compute(
        predictions=predictions,
        references=labels,
        average="macro"
    )

    return {
        "accuracy": acc["accuracy"],
        "macro_f1": macro_f1["f1"]
    }

# Training Arguments

In [43]:
training_args = TrainingArguments(
    output_dir="./lora-bert",

    eval_strategy="epoch",

    save_strategy="epoch",

    learning_rate=2e-4,

    per_device_train_batch_size=16,

    per_device_eval_batch_size=16,

    num_train_epochs=8,

    weight_decay=0.01,

    load_best_model_at_end=True,

    metric_for_best_model="macro_f1",

    logging_steps=100,

    report_to="none"
)

# Trainer

In [44]:
trainer = Trainer(
    model=model,

    args=training_args,

    train_dataset=train_dataset,

    eval_dataset=valid_dataset,

    compute_metrics=compute_metrics
)

In [45]:
trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1
1,0.776914,0.686942,0.866000,0.725845
2,0.632473,0.557647,0.892500,0.809309
3,0.551398,0.481613,0.905500,0.822188
4,0.476025,0.415135,0.913500,0.844362
5,0.445510,0.371043,0.921500,0.860095
6,0.393611,0.346527,0.923500,0.864931
7,0.366235,0.314717,0.933000,0.884502
8,0.311966,0.320666,0.930500,0.878139


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector

TrainOutput(global_step=2000, training_loss=0.4952310342788696, metrics={'train_runtime': 1364.2914, 'train_samples_per_second': 46.911, 'train_steps_per_second': 1.466, 'total_flos': 8506678050816000.0, 'train_loss': 0.4952310342788696, 'epoch': 8.0})

In [46]:
results = trainer.evaluate()

print(results)

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


{'eval_loss': 0.3147166967391968, 'eval_accuracy': 0.933, 'eval_macro_f1': 0.8845018755516328, 'eval_runtime': 17.5142, 'eval_samples_per_second': 114.193, 'eval_steps_per_second': 3.597, 'epoch': 8.0}


In [40]:
pred = trainer.predict(valid_dataset)

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


In [41]:
from sklearn.metrics import confusion_matrix, classification_report
import numpy as np

preds = np.argmax(pred.predictions, axis=1)

print(confusion_matrix(valid_df["label"], preds))
print()
print(classification_report(valid_df["label"], preds))

[[1588   12]
 [ 306   94]]

              precision    recall  f1-score   support

           0       0.84      0.99      0.91      1600
           1       0.89      0.23      0.37       400

    accuracy                           0.84      2000
   macro avg       0.86      0.61      0.64      2000
weighted avg       0.85      0.84      0.80      2000

